# LLM Power Law - Benchmarking Framework on Google Colab

This notebook helps you run LLM benchmarking experiments on Google Colab with free GPU access.

**Hardware:** Colab provides ~15GB VRAM (T4 GPU) - enough for models up to 13B with 4-bit quantization!

## Quick Start

1. **Enable GPU**: Runtime → Change runtime type → GPU (T4)
2. **Run all cells** in order
3. **Monitor progress** with built-in progress bars
4. **Download results** from the Files panel (left sidebar)

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Clone or update repository
import os

# Always work from /content directory in Colab
base_dir = "/content"
repo_url = "https://github.com/YOUR_USERNAME/LLMPowerLaw.git"
repo_name = "LLMPowerLaw"
repo_path = os.path.join(base_dir, repo_name)

# Change to base directory first
os.chdir(base_dir)
print(f"📂 Working directory: {os.getcwd()}")

if os.path.exists(repo_path):
    print(f"📂 Repository already exists. Updating to latest version...")
    os.chdir(repo_path)
    
    # Discard any local changes (config files modified by notebook cells)
    !git reset --hard HEAD
    !git clean -fd
    
    # Pull latest changes
    !git pull origin main
    print("✅ Repository updated to latest version!")
else:
    print(f"📥 Cloning repository for first time...")
    !git clone {repo_url}
    os.chdir(repo_path)
    print("✅ Repository cloned!")

print(f"✅ Current directory: {os.getcwd()}")
print(f"✅ Files: {os.listdir('.')[:10]}")  # Show first 10 files to verify

In [ ]:
# Install dependencies
print("📦 Installing core dependencies...")
!pip install -q torch transformers accelerate datasets

print("📦 Installing utilities...")
!pip install -q tqdm pyyaml python-dotenv pandas numpy scikit-learn

print("📦 Installing quantization support (4-bit models)...")
!pip install -q bitsandbytes

print("📦 Installing optional packages...")
!pip install -q sentencepiece --only-binary :all:

print("\n✅ Installation complete!")

## 2. Configure Models & Datasets

Colab's free T4 GPU (~15GB VRAM) can run:
- ✅ 7B models with 4-bit quantization (~4GB VRAM)
- ✅ 13B models with 4-bit quantization (~7GB VRAM)
- ✅ Multiple small models (2-3B)

**Pre-configured options below** - just uncomment what you want to test!

In [ ]:
# View current configuration
!cat config/models.yaml | head -n 100

In [ ]:
# Quick configuration for Colab (15GB VRAM)
# This enables models that work well on Colab's free tier

import yaml

# Read current config
with open('config/models.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL models first
for model in config['models']:
    model['enabled'] = False

# STEP 2: Enable only recommended models for Colab
models_to_enable = [
    'tinyllama-test',      # Fast testing (1.1B)
    'gemma-2b-4bit',       # Excellent quality (2B)
    'phi-3-mini-4bit',     # Balanced (3.8B)
    # 'llama-2-7b-4bit',     # High quality (7B) - Uncomment if needed
]

# Optionally enable for full Colab power (comment out if testing)
# models_to_enable.append('llama-2-13b-4bit')  # Needs ~7GB VRAM

for model in config['models']:
    if model['name'] in models_to_enable:
        model['enabled'] = True
        print(f"✅ Enabled: {model['name']}")

# Save config
with open('config/models.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Model configuration updated! ({len(models_to_enable)} models enabled)")

In [ ]:
# Configure datasets - start with small test
import yaml

with open('config/datasets.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL datasets first
for dataset in config['datasets']:
    dataset['enabled'] = False

# STEP 2: Enable only test dataset (5 samples - fast verification)
datasets_to_enable = [
    'custom_classification_test',  # 5 samples for quick test
    # 'sst2_test',                 # Uncomment for SST-2 test (10 samples)
]

for dataset in config['datasets']:
    if dataset['name'] in datasets_to_enable:
        dataset['enabled'] = True
        num_samples = dataset.get('num_samples', 'all')
        print(f"✅ Enabled: {dataset['name']} ({num_samples} samples)")

# Save
with open('config/datasets.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Dataset configuration updated! ({len(datasets_to_enable)} datasets enabled)")
print("\n💡 After test succeeds, enable more datasets or increase num_samples")

In [ ]:
# Configure prompting techniques
import yaml

with open('config/prompting_techniques.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL prompting techniques first
for technique in config['prompting_techniques']:
    technique['enabled'] = False

# STEP 2: Enable only specific techniques
techniques_to_enable = [
    'classification_simple',  # Custom classification prompt
    # 'zero_shot',            # Uncomment for zero-shot baseline
    # 'few_shot',             # Uncomment for few-shot examples
]

for technique in config['prompting_techniques']:
    if technique['name'] in techniques_to_enable:
        technique['enabled'] = True
        print(f"✅ Enabled technique: {technique['name']}")

# STEP 3: Update dataset-to-technique mapping (IMPORTANT!)
# This ensures each dataset uses the correct prompting technique
if 'global_settings' not in config:
    config['global_settings'] = {}

if 'dataset_technique_mapping' not in config['global_settings']:
    config['global_settings']['dataset_technique_mapping'] = {}

# Map datasets to their appropriate techniques
config['global_settings']['dataset_technique_mapping'].update({
    'custom_classification_test': ['classification_simple'],  # Test dataset
    'custom_classification': ['classification_simple'],        # Full dataset
    'sst2_test': ['classification_simple'],                   # SST-2 test
    'sst2': ['classification_simple'],                        # SST-2 full
    'mnli': ['classification_simple'],                        # MNLI
    'qqp': ['classification_simple'],                         # QQP
})

print(f"\n✅ Updated dataset→technique mappings:")
for dataset, techniques in config['global_settings']['dataset_technique_mapping'].items():
    if dataset in ['custom_classification_test', 'custom_classification', 'sst2_test', 'sst2', 'mnli', 'qqp']:
        print(f"   {dataset} → {techniques}")

# Save
with open('config/prompting_techniques.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Prompting configuration updated! ({len(techniques_to_enable)} techniques enabled)")
print("\n💡 Each dataset now has explicit technique mapping - no more fallback to zero_shot!")

## 3. Run Benchmark

This will:
1. Load each enabled model
2. Run predictions on test dataset
3. Show progress bars
4. Save results to `results/` folder

In [ ]:
# Run quick test (5 samples)
!python experiments/run_benchmark.py

## 4. Scale Up (After Test Succeeds)

Once the quick test works, increase sample size for real experiments:

In [ ]:
# Scale up to 100 samples
import yaml

with open('config/datasets.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable ALL datasets first
for dataset in config['datasets']:
    dataset['enabled'] = False

# STEP 2: Enable full datasets with more samples
datasets_to_enable = {
    'sst2': 100,        # Sentiment classification
    'mnli': 100,        # Natural language inference
    # 'qqp': 50,        # Question pairs (uncomment if needed)
}

for dataset in config['datasets']:
    if dataset['name'] in datasets_to_enable:
        dataset['enabled'] = True
        dataset['num_samples'] = datasets_to_enable[dataset['name']]
        print(f"✅ Enabled: {dataset['name']} ({datasets_to_enable[dataset['name']]} samples)")

with open('config/datasets.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"\n✅ Scaled up to full datasets! ({len(datasets_to_enable)} datasets enabled)")
print("💡 Increase sample counts in datasets_to_enable dict for larger experiments")

In [ ]:
# Run full experiment (will take longer)
!python experiments/run_benchmark.py

## 5. View Results

In [ ]:
# List all result files
!ls -lh results/

In [ ]:
# View latest summary
import json
import glob
from pathlib import Path

# Find latest summary file
summary_files = sorted(glob.glob('results/*_summary.json'))
if summary_files:
    latest = summary_files[-1]
    print(f"📊 Reading: {latest}\n")
    
    with open(latest, 'r', encoding='utf-8') as f:
        results = json.load(f)
    
    print(f"Experiment: {results['experiment_name']}")
    print(f"Total experiments: {len(results['experiments'])}")
    print("\nResults:")
    print("=" * 70)
    
    for exp in results['experiments']:
        if exp['status'] == 'completed':
            model = exp['model']
            dataset = exp['dataset']
            metrics = exp.get('metrics', {})
            accuracy = metrics.get('accuracy', 'N/A')
            print(f"✅ {model:<25} | {dataset:<15} | Accuracy: {accuracy}")
        else:
            print(f"❌ {exp['model']:<25} | {exp['dataset']:<15} | Failed")
else:
    print("No results found. Run the benchmark first!")

In [ ]:
# Create a zip file for easy download
!zip -r results.zip results/
print("\n✅ Results zipped! Download 'results.zip' from the Files panel (left sidebar)")

## 6. Advanced: Try Larger Models

Colab has enough VRAM for 13B models with 4-bit quantization:

In [ ]:
# Enable Llama 2 13B (uses ~7GB VRAM)
import yaml

with open('config/models.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# STEP 1: Disable all models first (optional - keeps existing config)
# Uncomment to start fresh:
# for model in config['models']:
#     model['enabled'] = False

# STEP 2: Enable 13B model (in addition to or instead of current models)
additional_models = [
    'llama-2-13b-4bit',  # High quality 13B model
]

for model in config['models']:
    if model['name'] in additional_models:
        model['enabled'] = True
        print(f"✅ Enabled: {model['name']} (expect ~3-5 min load time)")

with open('config/models.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\n⚠️  13B models are slower but produce higher quality results!")
print("💡 Uses ~7GB VRAM - well within Colab T4 limits")

## 💡 Tips for Colab

1. **Free tier limits**: 12-hour sessions, may disconnect if idle
2. **Save often**: Run with small samples first, then scale up
3. **Download results**: Files are deleted when session ends
4. **Monitor GPU**: Run `!nvidia-smi` in a cell to check usage
5. **Reduce samples**: If timeout, reduce `num_samples` in datasets

## 🚀 Next Steps

- Try different prompting techniques (see `config/prompting_techniques.yaml`)
- Add custom datasets (see `data_loaders/data/`)
- Compare multiple models on same dataset
- Export results and analyze locally